# Bronze to Silver Transformations

This notebook transforms raw Bronze tables into cleaned and conformed Silver Delta tables for the retail sales and inventory analytics project.

## Responsibilities

- standardize column names and data types
- trim and normalize source text
- parse currency and date fields
- enforce key and parsing quality checks before writing
- preserve source-file and ingestion metadata
- prevent duplicate sales records during incremental processing

## Update strategy

| Silver table | Grain | Write strategy |
|---|---|---|
| `silver_products` | One row per product | Overwrite |
| `silver_stores` | One row per store | Overwrite |
| `silver_inventory_position` | One row per store and product | Overwrite |
| `silver_calendar` | One row per date | Overwrite |
| `silver_sales` | One row per sale | Delta merge on `sale_id` |

Reference tables are replaced from complete source extracts. \
Sales use an incremental merge so that rerunning an ingestion batch does not create duplicate business records.


## 1. Setup

Import the PySpark functions used throughout the notebook.\
The default Lakehouse attached to this notebook is `lh_retail_analytics`.

In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 3, Finished, Available, Finished, False)

## 2. Products

Create one cleaned product record per `product_id`.

The source stores product cost and price as currency-formatted strings.\
The Silver transformation removes currency symbols and converts these fields to decimal values suitable for margin analysis.

### 2.1 Read Bronze products

In [2]:
bronze_products_df = spark.table("bronze_products")

display(bronze_products_df.limit(10))
bronze_products_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a4fb34af-4d40-4e2d-b1f5-37cb1777919c)

root
 |-- Product_ID: integer (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_Cost: string (nullable = true)
 |-- Product_Price: string (nullable = true)
 |-- _ingested_at: string (nullable = true)
 |-- _source_file: string (nullable = true)



### 2.2 Transform products

Standardize column names, trim descriptive fields, parse currency values, and preserve ingestion lineage.

In [3]:
silver_products_df = (
    bronze_products_df
    .select(
        F.col("Product_ID")
        .cast("int")
        .alias("product_id"),

        F.trim(F.col("Product_Name"))
        .alias("product_name"),

        F.trim(F.col("Product_Category"))
        .alias("product_category"),

        F.regexp_replace(
            F.trim(F.col("Product_Cost")),
            r"[$,]",
            "",
        )
        .cast("decimal(10, 2)")
        .alias("product_cost"),

        F.regexp_replace(
            F.trim(F.col("Product_Price")),
            r"[$,]",
            "",
        )
        .cast("decimal(10, 2)")
        .alias("product_price"),

        F.to_timestamp(F.col("_ingested_at"))
        .alias("ingested_at"),

        F.col("_source_file")
        .alias("source_file"),
    )
)

display(silver_products_df.limit(10))
silver_products_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b60bae0d-bdcd-4a34-873e-cc6ee017aa43)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_cost: decimal(10,2) (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



### 2.3 Validate products

Blocking checks prevent empty data, invalid keys, duplicate product records, and unparseable currency values from being written.

Products priced below cost are reported as a business warning rather than treated as invalid data.

In [4]:
product_quality = (
    silver_products_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("product_id").alias("distinct_product_ids"),

        F.sum(
            F.col("product_id").isNull().cast("int")
        ).alias("null_product_ids"),

        F.sum(
            F.col("product_cost").isNull().cast("int")
        ).alias("invalid_product_costs"),

        F.sum(
            F.col("product_price").isNull().cast("int")
        ).alias("invalid_product_prices"),

        F.sum(
            (
                F.col("product_price")
                < F.col("product_cost")
            ).cast("int")
        ).alias("prices_below_cost"),
    )
)

display(product_quality)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 90a65d7c-f050-4f70-a38e-83a884f99250)

In [5]:
product_quality_result = product_quality.first()

if product_quality_result["row_count"] == 0:
    raise ValueError("Product source is empty.")

if (
    product_quality_result["row_count"]
    != product_quality_result["distinct_product_ids"]
):
    raise ValueError("Product IDs are not unique.")

if product_quality_result["null_product_ids"] > 0:
    raise ValueError("Product IDs contain null values.")

if product_quality_result["invalid_product_costs"] > 0:
    raise ValueError("Some product costs could not be parsed.")

if product_quality_result["invalid_product_prices"] > 0:
    raise ValueError("Some product prices could not be parsed.")

print("Product blocking data-quality checks passed.")

print(
    "Products priced below cost:",
    product_quality_result["prices_below_cost"],
)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 7, Finished, Available, Finished, False)

Product blocking data-quality checks passed.
Products priced below cost: 0


### 2.4 Write and verify `silver_products`

The product source is a complete reference extract, so the Silver table is overwritten after all blocking checks pass.

In [6]:
(
    silver_products_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_products")
)

print("silver_products written successfully.")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 8, Finished, Available, Finished, False)

silver_products written successfully.


In [7]:
saved_products_df = spark.table("silver_products")

display(saved_products_df.orderBy("product_id").limit(10))
saved_products_df.printSchema()

print(f"Saved rows: {saved_products_df.count()}")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2ef6374-2625-4246-8bd6-d38404f61db6)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_cost: decimal(10,2) (nullable = true)
 |-- product_price: decimal(10,2) (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

Saved rows: 35


## 3. Stores

Create one cleaned store record per `store_id`.

The transformation standardizes store attributes and converts the source opening date to a proper date value.

### 3.1 Read Bronze stores

In [8]:
bronze_stores_df = spark.table("bronze_stores")

display(bronze_stores_df.limit(10))
bronze_stores_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1a84fc68-7a08-4779-9d92-d18a4f854545)

root
 |-- Store_ID: integer (nullable = true)
 |-- Store_Name: string (nullable = true)
 |-- Store_City: string (nullable = true)
 |-- Store_Location: string (nullable = true)
 |-- Store_Open_Date: string (nullable = true)
 |-- _ingested_at: string (nullable = true)
 |-- _source_file: string (nullable = true)



### 3.2 Transform stores

In [9]:
silver_stores_df = (
    bronze_stores_df
    .select(
        F.col("Store_ID")
        .cast("int")
        .alias("store_id"),

        F.trim(F.col("Store_Name"))
        .alias("store_name"),

        F.trim(F.col("Store_City"))
        .alias("store_city"),

        F.trim(F.col("Store_Location"))
        .alias("store_location"),

        F.to_date(
            F.col("Store_Open_Date"),
            "yyyy-MM-dd",
        ).alias("store_open_date"),

        F.to_timestamp(F.col("_ingested_at"))
        .alias("ingested_at"),

        F.col("_source_file")
        .alias("source_file"),
    )
)

display(silver_stores_df.orderBy("store_id").limit(10))
silver_stores_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e79d6476-b7b3-4571-90b5-c32430c4f993)

root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- store_open_date: date (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



### 3.3 Validate stores

In [10]:
store_quality = (
    silver_stores_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("store_id").alias("distinct_store_ids"),

        F.sum(
            F.col("store_id").isNull().cast("int")
        ).alias("null_store_ids"),

        F.sum(
            (
                F.col("store_name").isNull()
                | (F.length(F.col("store_name")) == 0)
            ).cast("int")
        ).alias("blank_store_names"),

        F.sum(
            F.col("store_open_date").isNull().cast("int")
        ).alias("invalid_open_dates"),
    )
)

display(store_quality)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c3d155e4-1686-44f8-a1a8-8367e98f5355)

In [11]:
store_quality_result = store_quality.first()

if store_quality_result["row_count"] == 0:
    raise ValueError("Store source is empty.")

if (
    store_quality_result["row_count"]
    != store_quality_result["distinct_store_ids"]
):
    raise ValueError("Store IDs are not unique.")

if store_quality_result["null_store_ids"] > 0:
    raise ValueError("Store IDs contain null values.")

if store_quality_result["blank_store_names"] > 0:
    raise ValueError("Store names contain blank values.")

if store_quality_result["invalid_open_dates"] > 0:
    raise ValueError("Some store opening dates could not be parsed.")

print("Store data-quality checks passed.")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 13, Finished, Available, Finished, False)

Store data-quality checks passed.


### 3.4 Write and verify `silver_stores`

In [12]:
(
    silver_stores_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_stores")
)

saved_stores_df = spark.table("silver_stores")

display(saved_stores_df.orderBy("store_id").limit(10))
saved_stores_df.printSchema()

print(f"Saved rows: {saved_stores_df.count()}")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e630a143-5d22-4945-a0b7-65da3fcd81bd)

root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- store_city: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- store_open_date: date (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

Saved rows: 50


## 4. Inventory Position

Create one current inventory-position record per store and product.

The source has no business snapshot date. The Silver table therefore
represents the inventory position supplied by the dataset and is not
described as historical inventory.

# 

### 4.1 Read Bronze inventory

In [13]:
bronze_inventory_df = spark.table("bronze_inventory")

display(bronze_inventory_df.limit(10))
bronze_inventory_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d6db1471-90e7-4343-baa0-c68653a9f744)

root
 |-- Store_ID: integer (nullable = true)
 |-- Product_ID: integer (nullable = true)
 |-- Stock_On_Hand: integer (nullable = true)
 |-- _ingested_at: string (nullable = true)
 |-- _source_file: string (nullable = true)



### 4.2 Transform inventory

In [14]:
silver_inventory_df = (
    bronze_inventory_df
    .select(
        F.col("Store_ID")
        .cast("int")
        .alias("store_id"),

        F.col("Product_ID")
        .cast("int")
        .alias("product_id"),

        F.col("Stock_On_Hand")
        .cast("int")
        .alias("stock_on_hand"),

        F.to_timestamp(F.col("_ingested_at"))
        .alias("ingested_at"),

        F.col("_source_file")
        .alias("source_file"),
    )
)

display(
    silver_inventory_df
    .orderBy("store_id", "product_id")
    .limit(10)
)

silver_inventory_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3a783fc2-62d8-4499-9452-67d56b1b15ca)

root
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- stock_on_hand: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



### 4.3 Validate keys and stock quantities

In [15]:
inventory_quality = (
    silver_inventory_df
    .agg(
        F.count("*").alias("row_count"),

        F.countDistinct(
            "store_id",
            "product_id",
        ).alias("distinct_store_product_keys"),

        F.sum(
            F.col("store_id").isNull().cast("int")
        ).alias("null_store_ids"),

        F.sum(
            F.col("product_id").isNull().cast("int")
        ).alias("null_product_ids"),

        F.sum(
            F.col("stock_on_hand").isNull().cast("int")
        ).alias("invalid_stock_values"),

        F.sum(
            (F.col("stock_on_hand") < 0).cast("int")
        ).alias("negative_stock_values"),

        F.sum(
            (F.col("stock_on_hand") == 0).cast("int")
        ).alias("zero_stock_positions"),
    )
)

display(inventory_quality)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e7d68c1b-7a15-45ec-8afe-6ed5f9f846d8)

In [16]:
inventory_orphan_store_count = (
    silver_inventory_df
    .join(
        spark.table("silver_stores").select("store_id"),
        on="store_id",
        how="left_anti",
    )
    .count()
)

inventory_orphan_product_count = (
    silver_inventory_df
    .join(
        spark.table("silver_products").select("product_id"),
        on="product_id",
        how="left_anti",
    )
    .count()
)

print("Orphan store keys:", inventory_orphan_store_count)
print("Orphan product keys:", inventory_orphan_product_count)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 18, Finished, Available, Finished, False)

Orphan store keys: 0
Orphan product keys: 0


In [17]:
inventory_quality_result = inventory_quality.first()

if inventory_quality_result["row_count"] == 0:
    raise ValueError("Inventory source is empty.")

if (
    inventory_quality_result["row_count"]
    != inventory_quality_result["distinct_store_product_keys"]
):
    raise ValueError(
        "Inventory store-product keys are not unique."
    )

if inventory_quality_result["null_store_ids"] > 0:
    raise ValueError("Inventory contains null store IDs.")

if inventory_quality_result["null_product_ids"] > 0:
    raise ValueError("Inventory contains null product IDs.")

if inventory_quality_result["invalid_stock_values"] > 0:
    raise ValueError(
        "Some inventory quantities could not be parsed."
    )

if inventory_quality_result["negative_stock_values"] > 0:
    raise ValueError(
        "Inventory contains negative stock quantities."
    )

if inventory_orphan_store_count > 0:
    raise ValueError(
        "Inventory contains unknown store IDs."
    )

if inventory_orphan_product_count > 0:
    raise ValueError(
        "Inventory contains unknown product IDs."
    )

print("Inventory blocking data-quality checks passed.")
print(
    "Explicit zero-stock positions:",
    inventory_quality_result["zero_stock_positions"],
)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 19, Finished, Available, Finished, False)

Inventory blocking data-quality checks passed.
Explicit zero-stock positions: 77


In [18]:
store_count = (
    spark.table("silver_stores")
    .select("store_id")
    .distinct()
    .count()
)

product_count = (
    spark.table("silver_products")
    .select("product_id")
    .distinct()
    .count()
)

expected_combination_count = store_count * product_count
available_combination_count = inventory_quality_result[
    "distinct_store_product_keys"
]
missing_combination_count = (
    expected_combination_count
    - available_combination_count
)

print("Expected store-product combinations:", expected_combination_count)
print("Available inventory positions:", available_combination_count)
print("Missing inventory positions:", missing_combination_count)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 20, Finished, Available, Finished, False)

Expected store-product combinations: 1750
Available inventory positions: 1593
Missing inventory positions: 157


### 4.4 Write and verify `silver_inventory_position`

In [19]:
(
    silver_inventory_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_inventory_position")
)

saved_inventory_df = spark.table(
    "silver_inventory_position"
)

display(
    saved_inventory_df
    .orderBy("store_id", "product_id")
    .limit(10)
)

print(f"Saved rows: {saved_inventory_df.count()}")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4f1877d7-96ed-42db-979c-31e2bf1a0110)

Saved rows: 1593


## 5. Calendar

Create one standardized calendar record per date. \
The source date format is parsed explicitly before the table is used to build the Gold date dimension.

### 5.1 Read Bronze calendar

In [20]:
bronze_calendar_df = spark.table("bronze_calendar")

display(bronze_calendar_df.limit(10))
bronze_calendar_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 83aaf981-dae8-4009-b97b-31624642f7fc)

root
 |-- Date: string (nullable = true)
 |-- _ingested_at: string (nullable = true)
 |-- _source_file: string (nullable = true)



### 5.2 Transform calendar

In [21]:
silver_calendar_df = (
    bronze_calendar_df
    .select(
        F.to_date(
            F.col("Date"),
            "M/d/yyyy",
        ).alias("calendar_date"),

        F.to_timestamp(F.col("_ingested_at"))
        .alias("ingested_at"),

        F.col("_source_file")
        .alias("source_file"),
    )
)

display(
    silver_calendar_df
    .orderBy("calendar_date")
    .limit(10)
)

silver_calendar_df.printSchema()

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1de838e-888f-4e04-9dfa-db68d12d6d30)

root
 |-- calendar_date: date (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



### 5.3 Validate calendar dates

In [22]:
calendar_quality = (
    silver_calendar_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            "calendar_date"
        ).alias("distinct_dates"),

        F.sum(
            F.col("calendar_date").isNull().cast("int")
        ).alias("invalid_dates"),

        F.min("calendar_date").alias("min_date"),
        F.max("calendar_date").alias("max_date"),
    )
)

display(calendar_quality)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9b9b658-2155-42a0-b77e-30117d2a0b49)

In [23]:
calendar_quality_result = calendar_quality.first()

if calendar_quality_result["row_count"] == 0:
    raise ValueError("Calendar source is empty.")

if (
    calendar_quality_result["row_count"]
    != calendar_quality_result["distinct_dates"]
):
    raise ValueError("Calendar dates are not unique.")

if calendar_quality_result["invalid_dates"] > 0:
    raise ValueError(
        "Some calendar dates could not be parsed."
    )

print("Calendar data-quality checks passed.")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 25, Finished, Available, Finished, False)

Calendar data-quality checks passed.


### 5.4 Write and verify `silver_calendar`

In [24]:
(
    silver_calendar_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_calendar")
)

saved_calendar_df = spark.table("silver_calendar")

display(
    saved_calendar_df
    .orderBy("calendar_date")
    .limit(10)
)

print(f"Saved rows: {saved_calendar_df.count()}")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 67e51fb8-77bc-49be-82c3-4440e8b9fe59)

Saved rows: 638


## 6. Sales

Create one cleaned transaction record per `sale_id`.

Sales are processed incrementally.\
Records are deduplicated by business key before being merged into the Silver Delta table, making repeated processing of the same source batch idempotent at the Silver layer.

### 6.1 Read Bronze sales

In [25]:
bronze_sales_df = spark.table("bronze_sales")

print(f"Bronze sales rows: {bronze_sales_df.count():,}")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 27, Finished, Available, Finished, False)

Bronze sales rows: 463,263


### 6.2 Standardize and deduplicate sales

In [26]:
standardized_sales_df = (
    bronze_sales_df
    .select(
        F.col("Sale_ID")
        .cast("long")
        .alias("sale_id"),

        F.to_date(
            F.col("Date"),
            "yyyy-MM-dd",
        ).alias("sale_date"),

        F.col("Store_ID")
        .cast("int")
        .alias("store_id"),

        F.col("Product_ID")
        .cast("int")
        .alias("product_id"),

        F.col("Units")
        .cast("int")
        .alias("units"),

        F.to_timestamp(F.col("_ingested_at"))
        .alias("ingested_at"),

        F.col("_source_file")
        .alias("source_file"),
    )
)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 28, Finished, Available, Finished, False)

In [27]:
sales_deduplication_window = (
    Window
    .partitionBy("sale_id")
    .orderBy(
        F.col("ingested_at").desc(),
        F.col("source_file").desc(),
    )
)

silver_sales_batch_df = (
    standardized_sales_df
    .withColumn(
        "row_number",
        F.row_number().over(
            sales_deduplication_window
        ),
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

duplicate_ingestion_rows = (
    standardized_sales_df.count()
    - silver_sales_batch_df.count()
)

print(
    "Duplicate Bronze ingestion rows removed:",
    duplicate_ingestion_rows,
)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 29, Finished, Available, Finished, False)

Duplicate Bronze ingestion rows removed: 0


### 6.3 Validate transaction and relationship rules

In [28]:
sales_quality = (
    silver_sales_batch_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("sale_id").alias(
            "distinct_sale_ids"
        ),

        F.sum(
            F.col("sale_id").isNull().cast("int")
        ).alias("null_sale_ids"),

        F.sum(
            F.col("sale_date").isNull().cast("int")
        ).alias("invalid_sale_dates"),

        F.sum(
            F.col("store_id").isNull().cast("int")
        ).alias("null_store_ids"),

        F.sum(
            F.col("product_id").isNull().cast("int")
        ).alias("null_product_ids"),

        F.sum(
            F.col("units").isNull().cast("int")
        ).alias("invalid_units"),

        F.sum(
            (F.col("units") <= 0).cast("int")
        ).alias("non_positive_units"),

        F.min("sale_date").alias("min_sale_date"),
        F.max("sale_date").alias("max_sale_date"),
        F.sum("units").alias("total_units"),
    )
)

display(sales_quality)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 35ddb51f-22a9-4f93-93d5-84455256cc36)

In [29]:
sales_orphan_store_count = (
    silver_sales_batch_df
    .join(
        spark.table("silver_stores").select("store_id"),
        on="store_id",
        how="left_anti",
    )
    .count()
)

sales_orphan_product_count = (
    silver_sales_batch_df
    .join(
        spark.table("silver_products").select("product_id"),
        on="product_id",
        how="left_anti",
    )
    .count()
)

print("Orphan store keys:", sales_orphan_store_count)
print("Orphan product keys:", sales_orphan_product_count)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 31, Finished, Available, Finished, False)

Orphan store keys: 0
Orphan product keys: 0


In [30]:
sales_quality_result = sales_quality.first()

if sales_quality_result["row_count"] == 0:
    raise ValueError("Sales source is empty.")

if (
    sales_quality_result["row_count"]
    != sales_quality_result["distinct_sale_ids"]
):
    raise ValueError(
        "Sales IDs remain duplicated after deduplication."
    )

if sales_quality_result["null_sale_ids"] > 0:
    raise ValueError("Sales contain null sale IDs.")

if sales_quality_result["invalid_sale_dates"] > 0:
    raise ValueError(
        "Some sales dates could not be parsed."
    )

if sales_quality_result["null_store_ids"] > 0:
    raise ValueError("Sales contain null store IDs.")

if sales_quality_result["null_product_ids"] > 0:
    raise ValueError("Sales contain null product IDs.")

if sales_quality_result["invalid_units"] > 0:
    raise ValueError(
        "Some sales quantities could not be parsed."
    )

if sales_quality_result["non_positive_units"] > 0:
    raise ValueError(
        "Sales contain non-positive quantities."
    )

if sales_orphan_store_count > 0:
    raise ValueError("Sales contain unknown store IDs.")

if sales_orphan_product_count > 0:
    raise ValueError("Sales contain unknown product IDs.")

print("Sales data-quality checks passed.")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 32, Finished, Available, Finished, False)

Sales data-quality checks passed.


### 6.4 Merge into `silver_sales`

In [31]:
if spark.catalog.tableExists("silver_sales"):
    silver_sales_table = DeltaTable.forName(
        spark,
        "silver_sales",
    )

    (
        silver_sales_table
        .alias("target")
        .merge(
            silver_sales_batch_df.alias("source"),
            "target.sale_id = source.sale_id",
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("silver_sales merged successfully.")

else:
    (
        silver_sales_batch_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("silver_sales")
    )

    print("silver_sales created successfully.")

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 33, Finished, Available, Finished, False)

silver_sales merged successfully.


### 6.5 Verify the saved table

In [32]:
saved_sales_df = spark.table("silver_sales")

saved_sales_quality = (
    saved_sales_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            "sale_id"
        ).alias("distinct_sale_ids"),
        F.min("sale_date").alias("min_sale_date"),
        F.max("sale_date").alias("max_sale_date"),
        F.sum("units").alias("total_units"),
    )
)

display(saved_sales_quality)

StatementMeta(, 37c729b4-2a23-4c01-9283-69aab73902ff, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6529af42-b2bf-4f04-aed4-76a6208cb99f)

## 7. Run Summary

A successful run produces the following Silver Delta tables:

- `silver_products`
- `silver_stores`
- `silver_inventory_position`
- `silver_calendar`
- `silver_sales`

Blocking transformation checks run before each table is written. Broader
cross-table and Gold-layer reconciliation checks are handled separately
by `nb_data_quality`.